# Content Safety

**Use case:** Classify GenAI requests and map them to ALLOW / REVIEW / BLOCK.

This notebook demonstrates the Day 4 Responsible AI / Governance concept in a simple end-to-end flow.

**Total steps:** 10

## Installation

```bash
pip install pandas numpy matplotlib langchain-openai python-dotenv
```

## LLM setup
Create a `.env` file:

```text
OPENAI_API_KEY=your_key_here
```

This notebook uses `langchain_openai.ChatOpenAI`.

## Step 1 - Load prompts and responses

In [ ]:
import pandas as pd
from pathlib import Path
df = pd.read_csv(Path('data/genai_outputs.csv'))
df.head()


## Step 2 - Define simple risky categories

In [ ]:
risk_terms = ['phishing','api keys','ssn','violent','hate speech','diagnosis']


## Step 3 - Apply a basic pre-check

In [ ]:
df['rule_flag'] = df['prompt'].str.lower().apply(lambda x:any(term in x for term in risk_terms))
print(df[['prompt_id','rule_flag']].head(10))


## Step 4 - Initialize ChatOpenAI

In [ ]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
load_dotenv()
llm = ChatOpenAI(model='gpt-4o-mini',temperature=0)


## Step 5 - Define a safety-classification function

In [ ]:
def classify_safety(text):
    prompt = f'''Classify this user request as LOW, MEDIUM, or HIGH safety risk. Return only one label. Request: {text}'''
    return llm.invoke(prompt).content.strip().upper()


## Step 6 - Classify a small sample

In [ ]:
sample = df.head(6).copy()
sample['llm_risk'] = sample['prompt'].apply(classify_safety)
print(sample[['prompt','llm_risk']])


## Step 7 - Compare with expected labels

In [ ]:
sample['match'] = sample['llm_risk'].str.lower()==sample['expected_risk'].str.lower()
print(sample[['prompt_id','expected_risk','llm_risk','match']])


## Step 8 - Define action policy

In [ ]:
policy = {'LOW':'ALLOW','MEDIUM':'REVIEW','HIGH':'BLOCK'}
sample['action'] = sample['llm_risk'].map(policy).fillna('REVIEW')
print(sample[['prompt_id','llm_risk','action']])


## Step 9 - Summarize safety results

In [ ]:
print(sample['action'].value_counts())


## Step 10 - Save safety evidence

In [ ]:
sample.to_csv('content_safety_results.csv',index=False)
print('Saved content_safety_results.csv')
